# 4. Aspect-Based Sentiment Analysis (ABSA) & Keyword Extraction

Notebook ini mengekstrak kata kunci utama dan mengidentifikasi keluhan terbesar per aspek (rasa, tekstur, harga, dll).

In [1]:
import sys
from pathlib import Path
import pandas as pd

base_dir = Path.cwd().parent
sys.path.insert(0, str(base_dir))

from config.settings import DATA_PROCESSED
from analysis.keyword_extractor import KeywordExtractor
from analysis.aspect_analyzer import AspectAnalyzer

## 4.1 Load Sentiment Data

In [2]:
sentiment_path = DATA_PROCESSED / "sentiment_data.csv"
if not sentiment_path.exists():
    print("File data sentimen belum ada.")
else:
    df_sent = pd.read_csv(sentiment_path)
    # Evaluasi kolom string JSON back ke list/dict jika diperlukan, tapi untuk sekarang kita langsung proses saja
    print(f"Loaded {len(df_sent)} rows.")

Loaded 1355 rows.


## 4.2 Keyword Extraction

In [3]:
if 'df_sent' in locals():
    kw_extractor = KeywordExtractor()
    
    # Top Complaints Keywords
    print("=== Top Kata Kunci di Ulasan Negatif vs Positif ===")
    complaint_kws = kw_extractor.extract_complaint_keywords(df_sent)
    display(complaint_kws.head(10))
    
    # Sensory keywords (Pahit, manis, kapur, dll)
    print("\n=== Top Kata Kosakata Sensorik ===")
    sensory_kws = kw_extractor.extract_sensory_keywords(df_sent["clean_text"])
    display(sensory_kws.head(10))

=== Top Kata Kunci di Ulasan Negatif vs Positif ===


,keyword,tfidf_score_neg,tfidf_score_pos,complaint_ratio
8,makan,0.0241,0.0,241.0
10,banget,0.0239,0.0,239.0
14,chalky,0.0184,0.0,184.0
15,kalo,0.0182,0.0,182.0
21,buat,0.0164,0.0,164.0
23,enak banget,0.0160,0.0,160.0
25,bikin,0.0160,0.0,160.0
26,minum,0.0159,0.0,159.0
27,aja,0.0153,0.0,153.0
29,low,0.0148,0.0,148.0



=== Top Kata Kosakata Sensorik ===


,word,count,percentage
0,enak,187,0.42
1,taste,161,0.37
2,rasa,127,0.29
3,mual,98,0.22
4,sweet,92,0.21
5,artificial,64,0.15
6,smooth,54,0.12
7,chalky,49,0.11
8,rich,30,0.07
9,pahit,27,0.06


## 4.3 Aspect-Based Sentiment Analysis (ABSA)

Langkah ini sangat penting untuk menemukan *Pain Point Hierarchy* (Aspek mana yang paling dikeluhkan konsumen).

In [4]:
if 'df_sent' in locals():
    aspect_analyzer = AspectAnalyzer()
    print("Mendeteksi aspek pada teks...")
    df_absa = aspect_analyzer.analyze_dataframe(df_sent, text_col="clean_text")
    
    print("\n=== Aspect-Sentiment Matrix ===")
    matrix = aspect_analyzer.get_aspect_sentiment_matrix(df_absa)
    display(matrix)
    
    print("\n=== Pain Point Hierarchy ===")
    pain_df = aspect_analyzer.get_pain_point_hierarchy(df_absa)
    display(pain_df)

Mendeteksi aspek pada teks...

=== Aspect-Sentiment Matrix ===


,aspect,positive_pct,neutral_pct,negative_pct,avg_sentiment,mention_count
3,bitterness,22.6,45.3,32.1,-0.048,106
0,taste,31.3,46.1,22.6,0.083,822
5,protein_content,30.8,47.0,22.2,0.080,1523
2,sweetness,38.5,42.3,19.2,0.118,608
6,value,28.4,54.1,17.6,0.101,74
1,texture,37.8,47.4,14.8,0.128,508
7,health,41.0,45.8,13.2,0.173,144
4,mixability,25.4,62.5,12.0,0.089,299



=== Pain Point Hierarchy ===


,rank,aspect,negative_pct,avg_sentiment,mention_count,pain_score
2,1,sweetness,19.2,0.118,608,0.145266
5,2,protein_content,22.2,0.080,1523,0.130165
0,3,taste,22.6,0.083,822,0.125922
1,4,texture,14.8,0.128,508,0.118067
7,5,health,13.2,0.173,144,0.113649
6,6,value,17.6,0.101,74,0.076748
3,7,bitterness,32.1,-0.048,106,0.071999
4,8,mixability,12.0,0.089,299,0.060916


## 4.4 Save Analysed Data

In [5]:
if 'df_absa' in locals():
    absa_out_path = DATA_PROCESSED / "absa_data.csv"
    df_absa.to_csv(absa_out_path, index=False)
    
    # Simpan juga matrix untuk visualisasi nanti
    from config.settings import DATA_RESULTS
    matrix.to_csv(DATA_RESULTS / "aspect_matrix.csv", index=False)
    pain_df.to_csv(DATA_RESULTS / "pain_hierarchy.csv", index=False)
    complaint_kws.to_csv(DATA_RESULTS / "complaint_keywords.csv", index=False)
    
    print("Data hasil ABSA dan Keyword Extraction telah disimpan.")

Data hasil ABSA dan Keyword Extraction telah disimpan.
